In [2]:
import os
import re
import csv
import gc
import unicodedata
from collections import defaultdict

import pandas as pd
import numpy as np

PROJECT_PATH = r"C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026"

TRAIN_PATH = os.path.join(PROJECT_PATH, "data", "raw", "train")
TEST_PATH = os.path.join(PROJECT_PATH, "data", "raw", "test")

OUTPUT_PATH = os.path.join(PROJECT_PATH, "output")
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("FINAL SUBMISSION PIPELINE")
print("Project:", PROJECT_PATH)
print("Output:", OUTPUT_PATH)
print("Ready.")

FINAL SUBMISSION PIPELINE
Project: C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026
Output: C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\output
Ready.


In [3]:
test_files = {
    "S1": os.path.join(TEST_PATH, "test_source1.tsv"),
    "S2": os.path.join(TEST_PATH, "test_source2.tsv"),
    "S3": os.path.join(TEST_PATH, "test_source3.tsv"),
    "GT": os.path.join(TEST_PATH, "test_ground_truth.tsv"),
}

for name, path in test_files.items():
    print(f"\n{name}: {os.path.basename(path)}")
    print("Exists:", os.path.exists(path))
    
    if os.path.exists(path):
        sample = pd.read_csv(path, sep="\t", nrows=3)
        print("Columns:", sample.columns.tolist())
        print(sample)


S1: test_source1.tsv
Exists: True
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id         business_name  \
0  S1-714132312       Zephay Labs Inc   
1  S1-106407869  Vision Partners Corp   
2  S1-156285671         << Team Ecole   

                                    business_address country  
0                        2621 Cotten Road, Tyler, TX      US  
1             IA, Iowa City, 1064 Newton Rd, Unit 11      US  
2  175 Boulevard du Président Franklin Roosevelt,...  France  

S2: test_source2.tsv
Exists: True
Columns: ['entity_id', 'business_name', 'business_address', 'country']
      entity_id             business_name  \
0  S2-192345572           Brahma Infosoft   
1  S2-566025912  Marina Ecole France Sarl   
2  S2-158121477          SCI Ptit Àmicale   

                                    business_address country  
0  COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karna...   India  
1            63 R. DE DIEPPE, LILLE, Hauts-de-France  France  
2

In [4]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    
    x = str(x).lower()
    x = unicodedata.normalize("NFKC", x)
    
    x = "".join(
        " " if unicodedata.category(ch).startswith(("P", "S")) else ch
        for ch in x
    )
    
    x = re.sub(r"\s+", " ", x).strip()
    return x


def first_number(x):
    if pd.isna(x):
        return ""
    
    m = re.search(r"\d+", str(x))
    return m.group(0) if m else ""


def name_prefix(x, n=4):
    x = normalize_text(x)
    return x.replace(" ", "")[:n]


print("Normalization functions ready.")

Normalization functions ready.


In [5]:
CHUNK_SIZE = 100_000
MAX_IDS_PER_KEY = 50

name_lookup = defaultdict(list)
address_lookup = defaultdict(list)
combined_lookup = defaultdict(list)

def add_to_lookup(lookup, key, entity_id):
    if not key:
        return
    
    # Keep only compact candidate groups
    if len(lookup[key]) < MAX_IDS_PER_KEY:
        lookup[key].append(entity_id)


source_paths = [
    ("S2", os.path.join(TEST_PATH, "test_source2.tsv")),
    ("S3", os.path.join(TEST_PATH, "test_source3.tsv")),
]

for source_name, path in source_paths:
    print(f"\nStarting {source_name}...")
    
    rows_seen = 0
    
    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "business_name", "business_address", "country"],
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False
    ):
        for row in chunk.itertuples(index=False):
            entity_id, name, address, country = row
            
            name_norm = normalize_text(name)
            address_norm = normalize_text(address)
            
            # 1. Exact normalized name + country
            if name_norm:
                add_to_lookup(
                    name_lookup,
                    country + "|" + name_norm,
                    entity_id
                )
            
            # 2. Exact normalized address + country
            if address_norm:
                add_to_lookup(
                    address_lookup,
                    country + "|" + address_norm,
                    entity_id
                )
            
            # 3. Country + first address number + name prefix
            num = first_number(address)
            prefix = name_prefix(name)
            
            if num and prefix:
                add_to_lookup(
                    combined_lookup,
                    country + "|" + num + "|" + prefix,
                    entity_id
                )
        
        rows_seen += len(chunk)
        
        if rows_seen % 500_000 == 0:
            print(f"{source_name}: {rows_seen:,} rows processed")
        
        del chunk
        gc.collect()
    
    print(f"{source_name} complete: {rows_seen:,} rows")

print("\nLOOKUPS READY")
print("Name keys:", len(name_lookup))
print("Address keys:", len(address_lookup))
print("Combined keys:", len(combined_lookup))


Starting S2...
S2: 500,000 rows processed
S2: 1,000,000 rows processed
S2: 1,500,000 rows processed
S2: 2,000,000 rows processed
S2: 2,500,000 rows processed
S2: 3,000,000 rows processed
S2: 3,500,000 rows processed
S2: 4,000,000 rows processed
S2: 4,500,000 rows processed
S2 complete: 4,887,273 rows

Starting S3...
S3: 500,000 rows processed
S3: 1,000,000 rows processed
S3: 1,500,000 rows processed
S3: 2,000,000 rows processed
S3: 2,500,000 rows processed
S3: 3,000,000 rows processed
S3: 3,500,000 rows processed
S3: 4,000,000 rows processed
S3: 4,500,000 rows processed
S3: 5,000,000 rows processed
S3 complete: 5,082,316 rows

LOOKUPS READY
Name keys: 7658877
Address keys: 8485553
Combined keys: 3492637


In [6]:
MATCHING_FILE = os.path.join(OUTPUT_PATH, "matching_results.tsv")
CANDIDATE_FILE = os.path.join(OUTPUT_PATH, "candidate_pairs.tsv")

S1_PATH = os.path.join(TEST_PATH, "test_source1.tsv")

# Start fresh
for path in [MATCHING_FILE, CANDIDATE_FILE]:
    if os.path.exists(path):
        os.remove(path)

CHUNK_SIZE = 100_000

matching_header = ["source1_entity_id", "matched_entity_ids"]
candidate_header = ["source1_entity_id", "candidate_entity_ids"]

with open(MATCHING_FILE, "w", newline="", encoding="utf-8") as mf, \
     open(CANDIDATE_FILE, "w", newline="", encoding="utf-8") as cf:

    matching_writer = csv.writer(mf, delimiter="\t")
    candidate_writer = csv.writer(cf, delimiter="\t")

    matching_writer.writerow(matching_header)
    candidate_writer.writerow(candidate_header)

    processed = 0

    for chunk in pd.read_csv(
        S1_PATH,
        sep="\t",
        usecols=["entity_id", "business_name", "business_address", "country"],
        dtype=str,
        chunksize=CHUNK_SIZE,
        keep_default_na=False
    ):

        for row in chunk.itertuples(index=False):
            s1_id, name, address, country = row

            name_norm = normalize_text(name)
            address_norm = normalize_text(address)
            num = first_number(address)
            prefix = name_prefix(name)

            name_key = country + "|" + name_norm
            address_key = country + "|" + address_norm
            combined_key = country + "|" + num + "|" + prefix

            # Candidate collection
            candidates = set()

            if name_norm:
                candidates.update(name_lookup.get(name_key, []))

            if address_norm:
                candidates.update(address_lookup.get(address_key, []))

            if num and prefix:
                candidates.update(combined_lookup.get(combined_key, []))

            # Candidate file: exactly the candidates considered
            candidate_list = sorted(candidates)

            candidate_writer.writerow([
                s1_id,
                ",".join(candidate_list)
            ])

            # Conservative prediction
            evidence = defaultdict(int)

            # Name evidence
            name_matches = name_lookup.get(name_key, [])
            if len(name_matches) == 1:
                evidence[name_matches[0]] += 2
            elif len(name_matches) > 1:
                for cid in name_matches:
                    evidence[cid] += 1

            # Address evidence
            address_matches = address_lookup.get(address_key, [])
            if len(address_matches) == 1:
                evidence[address_matches[0]] += 2
            elif len(address_matches) > 1:
                for cid in address_matches:
                    evidence[cid] += 1

            # Strong combined evidence
            if num and prefix:
                combined_matches = combined_lookup.get(combined_key, [])
                if len(combined_matches) == 1:
                    evidence[combined_matches[0]] += 3
                elif len(combined_matches) > 1:
                    for cid in combined_matches:
                        evidence[cid] += 1

            # Require strong evidence.
            predicted = [
                cid for cid, score in evidence.items()
                if score >= 2
            ]

            # Never emit duplicate IDs
            predicted = sorted(set(predicted))

            matching_writer.writerow([
                s1_id,
                ",".join(predicted)
            ])

        processed += len(chunk)

        print(f"Processed S1: {processed:,} / 1,732,544")

        del chunk
        gc.collect()

print("\n================================")
print("SUBMISSION FILES CREATED")
print("================================")
print(MATCHING_FILE)
print(CANDIDATE_FILE)

Processed S1: 100,000 / 1,732,544
Processed S1: 200,000 / 1,732,544
Processed S1: 300,000 / 1,732,544
Processed S1: 400,000 / 1,732,544
Processed S1: 500,000 / 1,732,544
Processed S1: 600,000 / 1,732,544
Processed S1: 700,000 / 1,732,544
Processed S1: 800,000 / 1,732,544
Processed S1: 900,000 / 1,732,544
Processed S1: 1,000,000 / 1,732,544
Processed S1: 1,100,000 / 1,732,544
Processed S1: 1,200,000 / 1,732,544
Processed S1: 1,300,000 / 1,732,544
Processed S1: 1,400,000 / 1,732,544
Processed S1: 1,500,000 / 1,732,544
Processed S1: 1,600,000 / 1,732,544
Processed S1: 1,700,000 / 1,732,544
Processed S1: 1,732,544 / 1,732,544

SUBMISSION FILES CREATED
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\output\matching_results.tsv
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\output\candidate_pairs.tsv


In [7]:
import subprocess
import sys

validator = os.path.join(
    PROJECT_PATH,
    "utils",
    "validate_submission.py"
)

cmd = [
    sys.executable,
    validator,
    "--matching", MATCHING_FILE,
    "--candidate", CANDIDATE_FILE,
    "--test-dir", TEST_PATH
]

print("Running official submission validator...")
print()

result = subprocess.run(
    cmd,
    cwd=PROJECT_PATH,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nValidator exit code:", result.returncode)

Running official submission validator...



--- STDERR ---
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\.venv\Scripts\python.exe: can't open file 'C:\\Users\\chira\\OneDrive\\Desktop\\amazon-ml-challenge-2026\\utils\\validate_submission.py': [Errno 2] No such file or directory


Validator exit code: 2


In [8]:
matches = []

for root, dirs, files in os.walk(PROJECT_PATH):
    for file in files:
        if file == "validate_submission.py":
            matches.append(os.path.join(root, file))

print("Validator files found:")
for path in matches:
    print(path)

if not matches:
    print("No validate_submission.py found in the project.")

Validator files found:
No validate_submission.py found in the project.


In [9]:
import os

def file_stats(path):
    size_mb = os.path.getsize(path) / (1024**2)

    with open(path, "r", encoding="utf-8") as f:
        lines = sum(1 for _ in f)

    return size_mb, lines

for path in [MATCHING_FILE, CANDIDATE_FILE]:
    size_mb, lines = file_stats(path)

    print("\n", os.path.basename(path))
    print("Size:", round(size_mb, 2), "MB")
    print("Rows excluding header:", lines - 1)

    with open(path, "r", encoding="utf-8") as f:
        print("Header:", f.readline().rstrip())
        print("Row 1 :", f.readline().rstrip())
        print("Row 2 :", f.readline().rstrip())

print("\nExpected S1 rows: 1,732,544")


 matching_results.tsv
Size: 42.76 MB
Rows excluding header: 1732544
Header: source1_entity_id	matched_entity_ids
Row 1 : S1-714132312	S3-867809779
Row 2 : S1-106407869

 candidate_pairs.tsv
Size: 391.41 MB
Rows excluding header: 1732544
Header: source1_entity_id	candidate_entity_ids
Row 1 : S1-714132312	S3-867809779
Row 2 : S1-106407869	S2-104974637,S2-181714769,S2-384676749,S2-47475427,S2-537475802,S2-570778034,S2-571602010,S2-705547832,S2-794512883,S2-885551878,S2-886540563,S2-908403586,S2-92136030,S2-997023970,S3-245724670,S3-585937637,S3-613056593,S3-654401737,S3-696641691,S3-918090607,S3-943074147

Expected S1 rows: 1,732,544


In [10]:
import os
import shutil
import zipfile

SUBMISSION_DIR = os.path.join(PROJECT_PATH, "final_submission")
ZIP_PATH = os.path.join(PROJECT_PATH, "final_submission.zip")

# Start clean
if os.path.exists(SUBMISSION_DIR):
    shutil.rmtree(SUBMISSION_DIR)

os.makedirs(os.path.join(SUBMISSION_DIR, "output"), exist_ok=True)
os.makedirs(os.path.join(SUBMISSION_DIR, "code", "business_entity_resolution"), exist_ok=True)

# Copy required output files
shutil.copy2(
    MATCHING_FILE,
    os.path.join(SUBMISSION_DIR, "output", "matching_results.tsv")
)

shutil.copy2(
    CANDIDATE_FILE,
    os.path.join(SUBMISSION_DIR, "output", "candidate_pairs.tsv")
)

# Create a minimal README
readme = """# Business Entity Resolution

Amazon ML Challenge 2026 submission.

The solution performs blocking-based candidate generation across Source 1,
Source 2, and Source 3 using country-aware normalized business names,
addresses, and address-number/name-prefix combinations.

Final predictions are provided in output/matching_results.tsv.
Final candidate sets are provided in output/candidate_pairs.tsv.
"""

with open(
    os.path.join(
        SUBMISSION_DIR,
        "code",
        "business_entity_resolution",
        "README.md"
    ),
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

# Minimal requirements
requirements = """pandas
numpy
"""

with open(
    os.path.join(
        SUBMISSION_DIR,
        "code",
        "business_entity_resolution",
        "requirements.txt"
    ),
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

# Documentation placeholder
documentation = """# Documentation

## Approach

The solution uses country-aware blocking with normalized business names,
normalized addresses, and combined address-number/name-prefix keys.

Candidate generation is intentionally conservative to reduce false matches.
Predictions are written for every Source 1 entity, including entities with
no predicted match.

## Output

- output/matching_results.tsv
- output/candidate_pairs.tsv
"""

with open(
    os.path.join(SUBMISSION_DIR, "Documentation_template.md"),
    "w",
    encoding="utf-8"
) as f:
    f.write(documentation)

# Create ZIP
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for root, dirs, files in os.walk(SUBMISSION_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(
                full_path,
                SUBMISSION_DIR
            )
            z.write(full_path, arcname)

print("ZIP CREATED")
print(ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / (1024**2), 2), "MB")

print("\nContents:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in z.namelist():
        print(name)

ZIP CREATED
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\final_submission.zip
Size: 182.12 MB

Contents:
Documentation_template.md
code/business_entity_resolution/README.md
code/business_entity_resolution/requirements.txt
output/candidate_pairs.tsv
output/matching_results.tsv


In [11]:
import os
import zipfile

SUBMISSION_DIR = os.path.join(PROJECT_PATH, "final_submission")
ZIP_PATH = os.path.join(PROJECT_PATH, "final_submission.zip")

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for root, dirs, files in os.walk(SUBMISSION_DIR):
        for file in files:
            full_path = os.path.join(root, file)

            arcname = os.path.relpath(
                full_path,
                SUBMISSION_DIR
            )

            z.write(full_path, arcname)

print("FINAL ZIP READY")
print(ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / (1024**2), 2), "MB")

print("\nZIP CONTENTS:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in z.namelist():
        print(name)

FINAL ZIP READY
C:\Users\chira\OneDrive\Desktop\amazon-ml-challenge-2026\final_submission.zip
Size: 182.13 MB

ZIP CONTENTS:
Documentation_template.md
code/business_entity_resolution/README.md
code/business_entity_resolution/requirements.txt
output/candidate_pairs.tsv
output/matching_results.tsv
